In [ ]:
import pandas as pd
from taxomind.utils.taxonomy_utils import (transform_taxonomy_to_training_format, 
                                           process_taxonomy_parent,
                                           identify_level,
                                           jsonize_taxonomy)


In [ ]:
TAXONOMY_KEY = "ISCO"

In [ ]:
%load_ext kedro.ipython
%reload_kedro 

In [ ]:
clean_taxonomy = "isco_clean_taxonomy"

if TAXONOMY_KEY == 'ISCO':
    dataset_name = 'isco_taxonomy_definition'
    column_mapping = {"Level":"level","ISCO 08 Code": "code", 
                        'Title EN':'label', 'Definition':'definition', 
                        'Tasks include':"examples"}
    LEVEL_NAMES = {
        "1": "Major group",
        "2": "Sub-major group",
        "3": "Minor group",
        "4": "Unit group",
        }
    
elif TAXONOMY_KEY == 'ISIC':
    dataset_name = 'isic_taxonomy_definition'
    column_mapping = {"Code": "code", 
                        'ISIC Rev. 4 label':'label', 
                        'Inclusions': 'examples', 
                        }
    LEVEL_NAMES = {
        "1": "Section",
        "2": "Division",
        "3": "Group",
        "4": "Class",
    }

In [ ]:
df = catalog.load(dataset_name)
selected_columns = list(column_mapping.values())
df.rename(columns=column_mapping, inplace=True)
if TAXONOMY_KEY == 'ISIC':
    df['definition'] = df['examples']
    df['level'] = df['code'].apply(identify_level)
    selected_columns += ["definition",'level']

In [ ]:
df = df[selected_columns]

In [ ]:
df = process_taxonomy_parent(df)

In [ ]:
df

In [ ]:
text_to_remove = ['This section includes',
'This division includes',
'This group includes',
'This class includes',
'Tasks performed by',"Tasks include -", 
"Tasks performed usually include:",
"In such cases tasks would include -"]

In [ ]:
mask1 = df['examples'].str.contains('See class', na=False) 
mask2 = df['examples'].str.contains('See division', na=False)
mask3 = pd.isnull(df['examples']) | (df['examples'].str.strip() == '')
df.loc[mask1 | mask2 | mask3, 'examples'] = df.loc[mask1 | mask2 | mask3, 'label']

In [ ]:
df['examples'] = df['examples'].str.replace('|'.join(text_to_remove), '', regex=True)
df['examples'] = df['examples'].str.replace(r'[-*:]', '', regex=True)
df["examples"] = df["examples"].str.replace("matÃÆÃÂ©", "mate", regex=False)
#df["examples"] = df["examples"].str.replace("\n", "\\n", regex=False)
df['examples'] = df['examples'].str.strip()
if TAXONOMY_KEY == 'ISIC':
    df['definition'] = df['examples'].copy()

In [ ]:
##NOTE:
## Hack FOR Asdreas
# df['id'] = df['code']
# df.rename(columns={"parentCode":"parent_id"}, inplace=True)
# df = df[['id', "label",'parent_id','level','definition']]

In [ ]:
catalog.save('taxonomy_definition', {TAXONOMY_KEY: df})

In [ ]:
taxo_json = jsonize_taxonomy(df,TAXONOMY_KEY, LEVEL_NAMES)

In [ ]:
catalog.save('taxonomy_request', {TAXONOMY_KEY: taxo_json})

In [ ]:

df_ext = transform_taxonomy_to_training_format(df)
df_ext.drop_duplicates(subset='text',keep=False, inplace=True)
df_ext['taxonomyKey'] = TAXONOMY_KEY

In [ ]:
df_ext

In [ ]:
catalog.save('taxonomy_training', {TAXONOMY_KEY: df_ext})

In [ ]:
df = catalog.load('taxonomy_training')